In [22]:
# !pip install tqdm
#!pip install --upgrade langchain-openai
# !pip install boto3
# !pip install lxml


In [23]:
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

import re
import pandas as pd
import numpy as np
import ast
from xml.dom import minidom

import getpass
import os
from dotenv import load_dotenv
from tqdm import tqdm
import tiktoken
import time
import json
import boto3
from botocore.exceptions import ClientError
import xml.etree.ElementTree as ET
import ast
from xml.dom import minidom
from lxml import etree
import json
from ast import literal_eval


In [24]:
# Load environment variables from .env file
load_dotenv()

# Set them in the OS environment
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


session = boto3.Session(
    aws_access_key_id='',
    aws_secret_access_key='',
    region_name=''
)



In [25]:
# Getting Sentences with Explanation.
# generated_sentence_final= pd.read_excel('/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Jodi_Draft_TransformationEditsv01_Sentence.xlsx')
generated_sentence_final= pd.read_excel('/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Jodi_Draft_TransformationEditsv01_NoSentence.xlsx')
# generated_sentence_final= generated_sentence_final[generated_sentence_final['Model_Transformation'] == 'us.deepseek.r1-v1:0']
generated_sentence_final= generated_sentence_final[generated_sentence_final['Model_Transformation'] == 'gpt-4.1-2025-04-14']
# generated_sentence_final= generated_sentence_final.head(5)
generated_sentence_final

,Title,Paragraph,Generated_Sentence,Provenance,Id_Order,Has_Provenance,Source_Details,Id_Order_Provenance,Id_Provenance,Final_Sentence,Response_Transformation,Explanation_Transformation,Thinking_Transformation,Is_Mixed_Model_Transformation,Context_Window_Transformation,Time_Transformation,Model_Transformation
0,1. Citation of Retracted Publications and Epis...,1,Research on the citation of retracted publicat...,"(339,1)",1,Yes,"[{'Provenance': '(339,1)', 'Paper_Title': 'Add...",1.0,(Prov.1),Research on the citation of retracted publicat...,<response> \n <transformation_explanation> ...,"Provenance(339,1): Paraphrasing and Copy-Paste...",NaN,NO,402,1.41,gpt-4.1-2025-04-14
1,1. Citation of Retracted Publications and Epis...,1,"In parallel, several citation management tools...","(339,12)",2,Yes,"[{'Provenance': '(339,12)', 'Paper_Title': 'Ad...",2.0,(Prov.2),"In parallel, several citation management tools...",<response> \n <transformation_explanation> ...,"Provenance(339,12): Copy-Paste with minor edits.",NaN,NO,368,3.20,gpt-4.1-2025-04-14
2,1. Citation of Retracted Publications and Epis...,1,Post-retraction citation was first studied in ...,NaN,3,No,NaN,NaN,NaN,Post-retraction citation was first studied in ...,NaN,NaN,NaN,NaN,0,0.00,gpt-4.1-2025-04-14
3,1. Citation of Retracted Publications and Epis...,1,g,NaN,4,No,NaN,NaN,NaN,g,NaN,NaN,NaN,NaN,0,0.00,gpt-4.1-2025-04-14
4,1. Citation of Retracted Publications and Epis...,1,", [212] [41] [10]); the most comprehensive ide...","(8,5)",5,Yes,"[{'Provenance': '(8,5)', 'Paper_Title': 'Conti...",3.0,(Prov.3),", [212] [41] [10]); the most comprehensive ide...",<response> \n <transformation_explanation> ...,"- Provenance(8,5): Copy-Paste with minor edits.",NaN,NO,468,4.55,gpt-4.1-2025-04-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,3. Argumentation Theory and Curation in Scient...,2,One well-known example of provenance-based cur...,"(45,13)",62,Yes,"[{'Provenance': '(45,13)', 'Paper_Title': 'Tow...",50.0,(Prov.50),One well-known example of provenance-based cur...,<response> \n <transformation_explanation> ...,"Provenance(45,13): Copy-Paste with minor edits.",NaN,NO,384,57.75,gpt-4.1-2025-04-14
62,3. Argumentation Theory and Curation in Scient...,2,Argument scheme-based curation approaches rely...,"(45,17) (45,18)",63,Yes,"[{'Provenance': '(45,17)', 'Paper_Title': 'Tow...",51.0,(Prov.51),Argument scheme-based curation approaches rely...,<response> \n <transformation_explanation> ...,"Provenance(45,17): Copy-Paste with minor edits...",NaN,NO,444,58.93,gpt-4.1-2025-04-14
63,3. Argumentation Theory and Curation in Scient...,2,Such argument schemes document the underlying ...,"(45,19)",64,Yes,"[{'Provenance': '(45,19)', 'Paper_Title': 'Tow...",52.0,(Prov.52),Such argument schemes document the underlying ...,<response> \n <transformation_explanation> ...,"Provenance(45,19): Copy-Paste with minor edits.",NaN,NO,376,59.83,gpt-4.1-2025-04-14
64,3. Argumentation Theory and Curation in Scient...,3,Argument schemes describe stereotypical reason...,"(45,2)",65,Yes,"[{'Provenance': '(45,2)', 'Paper_Title': 'Towa...",53.0,(Prov.53),Argument schemes describe stereotypical reason...,<response> \n <transformation_explanation> ...,"Provenance(45,2): Copy-Paste with minor edits.",NaN,NO,350,61.26,gpt-4.1-2025-04-14


In [26]:
# generated_sentence_final.iloc[3].Generated_Sentence  

In [27]:
def generate_xml_sentence(df):
    """
    Generates XML content in the exact specified format without any wrapping tags.
    Handles both string-encoded and already-parsed Source_Details.
    """
    xml_lines = []
    
    for _, row in df.iterrows():
        # Add generated sentence
        xml_lines.append('<generated_sentence>')
        xml_lines.append(str(row["Generated_Sentence"]))
        xml_lines.append('</generated_sentence>')
        xml_lines.append('\n')


        # xml_lines.append(f'<generated_sentence>{row["Generated_Sentence"]}</generated_sentence>')

        
        xml_lines.append('<source_details>')
        
        try:
            # Handle both string and already-parsed cases
            details = row['Source_Details']
            
            # If it's a string, parse it
            if isinstance(details, str):
                try:
                    details = json.loads(details.replace("'", '"'))
                except json.JSONDecodeError:
                    details = literal_eval(details)
            
            # Ensure we have a list (even if single dict)
            if not isinstance(details, list):
                details = [details]
                
            for detail in details:
                xml_lines.append('   <source>')
                xml_lines.append(f'      <provenance>{detail.get("Provenance", "")}</provenance>')
                xml_lines.append(f'      <text>{detail.get("Sentence", "")}</text>')
                xml_lines.append('   </source>')
                
        except Exception as e:
            print(f"Error processing Source_Details: {e}")
            continue
            
        xml_lines.append('</source_details>')
    
    return '\n'.join(xml_lines)

In [28]:
def extract_think_text(text):
    match = re.search(r"<think>(.*?)</think>(.*)", text, re.DOTALL)
    if match:
        between_think = match.group(1).strip()
        after_think = match.group(2).strip()
    else:
        between_think = ""
        after_think = ""
    
    return between_think, after_think

In [29]:
# Put it the message response :    - If all transformations are supported, write "These transformations ensure that the generated sentence accurately reflects the original sources, making it trustworthy.".
prompt = f"""
<context>  
You are an expert in citation analysis. Your task is to judge whether provenances contribute full, partial, or no support for their associated generated sentence.  
</context>  

_content_

<citation_criteria>  
Full Support: All information in the generated sentence is directly supported by one or more provenances. This includes:  
- Explicit statements in the provenance(s).  
- Clear, unambiguous implications from the provenance(s).  
- Reasonable inferences that logically follow from the provenance(s).  

Partial Support: Some parts of the generated sentence meet Full Support criteria, while others:  
- Lack direct provenance support.  
- Represent unreasonable inferences.  
- Involve citation misplacement or overcitation . 

No Support: The provenance(s) do not support any part of the generated sentence, including:  
- Contradictions with the source.  
- Completely unsupported claims.  
- Unreasonable inferences with no basis in the provenance(s).  
</citation_criteria>  

<citation_issues>  
Citation Misplacement: Citations are technically present but incorrectly positioned, distorting their intended link to the text in the provenance.This includes:
    - Citing a source for a claim it does not support.
    - Placing a citation such that it appears to support the wrong phrase or clause.
    - Displacing relevant citations from their appropriate context.
Overcitation: Citations are included where they don't belong because they support a different claim or are already implied by another citation.  
Unreasonable Inference: Claims that cannot be logically derived from the provenance(s) without additional assumptions.  
</citation_issues>  

<task_requirements>  
1. Analyze the generated sentence by:  
   - Identifying which parts come from which provenance(s).  
   - Distinguishing between:  
     * Direct support (explicit statements).  
     * Reasonable inferences (logical derivations). 
     * Unsupported claims.  
   - Evaluating citation placement accuracy, including citation misplacement. 

2. Output format (XML):  
<response>  
   <citation_precision>[Full/Partial/No Support]</citation_precision>  
   <citation_explanation>  
      [1) Overall trustworthiness assessment:  
       - For Full Support: "The generated sentence accurately reflects the original sources, making it trustworthy." . 
       - For Partial/No Support: Identify specific issues (citation problems vs. unreasonable inferences).
      
      2) Detail support for each claim, specifying:  
       - Direct support (provenance text).  
       - Reasonable inferences (explain derivation).  
       - Unsupported aspects (if any).
       - Flag and explain citation misplacement, including what was cited incorrectly and where it should have been placed.]  
   </citation_explanation>  
</response>  

3. Key Rules:  
- Treat reasonable inferences as supported when they logically follow from provenances.  
- Explicitly flag unreasonable inferences requiring direct evidence.  
- For Partial Support, specify which claims lack support and why.
- Distinguish between citation placement errors (e.g., misplacement, overcitation) and content support issues.
- **Do not include special XML characters in the output (e.g. &, <, >,",')**.

4. Examples of Citation Misplacement:
- Generated: Entrepreneurial orientation (EO) is widely recognized as a critical factor for the survival, growth, and innovation of family firms [193] [73]...
  Correct: ...survival, growth [193][73], and innovation... (the citations are misaligned and misleadingly suggest support for "innovation").
- Generated: ...through architectures like pointer-generator networks and transformer models [29][31][35][41][17]...
  Correct: ...networks and transformer models [29][41][17]... (extra citations [31][35] are misapplied).
- Generated: ...human writing processes [4][39][257]...
  Correct: ...[4][39]... (citation [257] is misplaced; it supports a different part of the sentence).
 
</task_requirements>   
"""

# prompt = f"""
# <context>  
# You are an expert in citation analysis. Your task is to judge whether provenances contribute full, partial, or no support for their associated generated sentence.  
# </context>  

# _content_

# <citation_criteria>  
# Full Support: All information in the generated sentence is directly supported by one or more provenances. This includes:  
# - Explicit statements in the provenance(s).  
# - Clear, unambiguous implications from the provenance(s).  
# - Reasonable inferences that logically follow from the provenance(s).  

# Partial Support: Some parts of the generated sentence meet Full Support criteria, while others:  
# - Lack direct provenance support.  
# - Represent unreasonable inferences.  
# - Involve citation misplacement or overcitation.  

# No Support: The provenance(s) do not support any part of the generated sentence, including:  
# - Contradictions with the source.  
# - Completely unsupported claims.  
# - Unreasonable inferences with no basis in the provenance(s).  
# </citation_criteria>  

# <citation_issues>  
# Citation Misplacement: Citations are technically present but incorrectly positioned, distorting their intended link to the text in the provenance. This includes:  
#     - Citing a source for a claim it does not support.  
#     - Placing a citation such that it appears to support the wrong phrase or clause.  
#     - Displacing relevant citations from their appropriate context.  
# Overcitation: Citations are included where they don't belong because they support a different claim or are already implied by another citation.  
# Unreasonable Inference: Claims that cannot be logically derived from the provenance(s) without additional assumptions.  
# </citation_issues>  

# <task_requirements>  
# 1. Analyze the generated sentence by:  
#    - Identifying which parts come from which provenance(s).  
#    - Distinguishing between:  
#      * Direct support (explicit statements).  
#      * Reasonable inferences (logical derivations).  
#      * Unsupported claims.  
#    - Evaluating citation placement accuracy, including citation misplacement.  

# 2. Output format (XML):  
# <response>  
#    <citation_precision>[Full/Partial/No Support]</citation_precision>  
#    <citation_explanation>  
#       [1) Overall trustworthiness assessment:  
#        - For Full Support: "The generated sentence accurately reflects the original sources, making it trustworthy."  
#        - For Partial/No Support: Identify specific issues (citation problems vs. unreasonable inferences).  

#       2) Detail support for each claim, specifying:  
#        - Direct support (quote or paraphrase provenance text safely).  
#        - Reasonable inferences (explain derivation).  
#        - Unsupported aspects (if any).  
#        - Flag and explain citation misplacement, including what was cited incorrectly and where it should have been placed.  

#       **Important:** Do not copy raw provenance text directly into the XML. If quoting, escape all XML special characters so the output remains valid.  
#    </citation_explanation>  
# </response>  

# 3. Key Rules:  
# - Treat reasonable inferences as supported when they logically follow from provenances.  
# - Explicitly flag unreasonable inferences requiring direct evidence.  
# - For Partial Support, specify which claims lack support and why.  
# - Distinguish between citation placement errors (e.g., misplacement, overcitation) and content support issues.  
# - Escape all special XML characters in the output to ensure valid XML:  
#    * & → &amp;  
#    * < → &lt;  
#    * > → &gt;  
#    * " → &quot;  
#    * ' → &apos;  
# - Never insert raw, unescaped provenance text inside `<citation_explanation>`. Always paraphrase or escape.  

# 4. Examples of Citation Misplacement:  
# - Generated: Entrepreneurial orientation (EO) is widely recognized as a critical factor for the survival, growth, and innovation of family firms [193] [73]...  
#   Correct: ...survival, growth [193][73], and innovation... (the citations are misaligned and misleadingly suggest support for "innovation").  
# - Generated: ...through architectures like pointer-generator networks and transformer models [29][31][35][41][17]...  
#   Correct: ...networks and transformer models [29][41][17]... (extra citations [31][35] are misapplied).  
# - Generated: ...human writing processes [4][39][257]...  
#   Correct: ...[4][39]... (citation [257] is misplaced; it supports a different part of the sentence).  
# </task_requirements>  

# """

In [30]:
# def extract_explanation(xml_string):
#     # Parse the XML content
#     print("xml_string: \n")
#     print(xml_string)

#     root = ET.fromstring(xml_string)

#     # Extract the synthesis_explanation text
#     synthesis_explanation = root.find('citation_explanation').text.strip()

#     return synthesis_explanation






# def extract_citation_precision(xml_string):
#     # Parse the XML content
#     root = ET.fromstring(xml_string)
    
#     # Extract the synthesis_explanation text
#     citation_precision = root.find('citation_precision').text.strip()

#     return citation_precision




    # Regex to strip illegal control characters
_INVALID_XML_CHARS = re.compile(
    u'[\x00-\x08\x0b-\x0c\x0e-\x1f\uFFFE\uFFFF]'
)

def _sanitize_xml(s: str) -> str:
    """Clean up typical bad tokens in model output before parsing."""
    s = s.strip()
    # Remove illegal control chars
    s = re.sub(_INVALID_XML_CHARS, "", s)
    # Escape stray '&' that are not entities
    s = re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#\d+;)', '&amp;', s)
    return s







def extract_explanation(xml_string: str) -> str:
    """Safely parse XML and extract <extract_explanation> text."""
    s = _sanitize_xml(xml_string)

    try:
        root = ET.fromstring(s)
    except ET.ParseError:
        # If multiple roots or junk outside tags, wrap in <root>
        try:
            root = ET.fromstring(f"<root>{s}</root>")
        except ET.ParseError:
            # If still invalid, just return raw string as fallback
            return s.strip()

    node = root.find('.//citation_explanation')

    if node is not None and node.text:
        return node.text.strip()
    else:
        # Fallback: return whole text content
        return ET.tostring(root, encoding="unicode", method="text").strip()




def extract_citation_precision(xml_string: str) -> str:
    """Safely parse XML and extract <textract_citation_precision> text."""
    s = _sanitize_xml(xml_string)

    try:
        root = ET.fromstring(s)
    except ET.ParseError:
        # If multiple roots or junk outside tags, wrap in <root>
        try:
            root = ET.fromstring(f"<root>{s}</root>")
        except ET.ParseError:
            # If still invalid, just return raw string as fallback
            return s.strip()

    node = root.find('.//citation_precision')

    if node is not None and node.text:
        return node.text.strip()
    else:
        # Fallback: return whole text content
        return ET.tostring(root, encoding="unicode", method="text").strip()



In [31]:
def get_explanation_others(df: pd.DataFrame, model_name: str, model_mixed: str):
    model_mixed = model_mixed.upper()

    if model_mixed == "YES" or model_mixed == "NO": 
        start_time = time.time()  # Start timing

        # Initialize the chat model based on the specified model name
        if "gpt" in model_name:
            llm_generator = ChatOpenAI(model_name=model_name, temperature=0)
            if "gpt-4o" in model_name:
                encoding = tiktoken.encoding_for_model(model_name)
            else:
                encoding = tiktoken.get_encoding("cl100k_base")
        else:
            if "o3-mini" in model_name:
                llm_generator = ChatOpenAI(model_name=model_name, temperature=None)
                encoding = tiktoken.get_encoding("cl100k_base")
            else:
                llm_generator = ChatOllama(model=model_name, temperature=0)
                encoding = tiktoken.get_encoding("cl100k_base")
        
        
        for i in range(len(df)):
            row_df = df.iloc[[i]]

            if row_df["Has_Provenance"].values[0] == "No":
                df.at[i, 'Response_Citation'] = "N/A"
                df.at[i, 'Explanation_Citation'] = "N/A"
                df.at[i, 'Citation_Precision'] = "N/A"
                #df.at[i, 'Clauses'] = "N/A"
                df.at[i, 'Thinking_Citation'] = "N/A"
                df.at[i, 'Is_Mixed_Model_Citation'] = "N/A"
                df.at[i, 'Context_Window_Citation'] = 0
                df.at[i, 'Time_Citation'] = 0
                df.at[i, 'Model_Citation'] = model_name
            else:
                # Make sure 'prompt' is defined before using this
                user_question = prompt.replace("_content_", generate_xml_sentence(row_df))

                # Calculate number of tokens
                num_tokens = len(encoding.encode(user_question))
                print(f"\n Model Name: {model_name} - Context Window: {num_tokens} tokens \n")
                print(user_question)

                # Message formatting based on model type
                if "gpt" in model_name or "o3-mini" in model_name:
                    messages = [
                        {"role": "system", "content": "You are an expert in citation analysis. Your task is to judge whether provenances contribute full, partial, or no support for their associated **generated sentence**."},
                        {"role": "user", "content": user_question}
                    ]
                else:
                    template = ChatPromptTemplate.from_messages([
                        ("system", "You are an expert in citation analysis. Your task is to judge whether provenances contribute full, partial, or no support for their associated **generated sentence**."),
                        ("human", "{user_input}")
                    ])
                    messages = template.format_messages(user_input=user_question)

                # Invoke model
                response = llm_generator.invoke(messages)
                response_temp = response.content

                # Separate thinking process and final answer
                if "gpt" in model_name or "o3-mini" in model_name:
                    thinking = "NA"
                    response_final = response_temp  
                else:
                    thinking, response_final = extract_think_text(response_temp)

                end_time = time.time()
                elapsed_time = round(end_time - start_time, 2)

                print(f"\nTotal execution time: {elapsed_time} seconds")
                print("Response: \n", response_final)

                # Assign response values to current row
                df.at[i, 'Response_Citation'] = response_final
                df.at[i, 'Explanation_Citation'] = extract_explanation(response_final)
                df.at[i, 'Citation_Precision'] = extract_citation_precision(response_final)
                #df.at[i, 'Clauses'] = extract_provenance_clauses(response_final)
                df.at[i, 'Thinking_Citation'] = thinking
                df.at[i, 'Is_Mixed_Model_Citation'] = model_mixed
                df.at[i, 'Context_Window_Citation'] = num_tokens
                df.at[i, 'Time_Citation'] = elapsed_time
                df.at[i, 'Model_Citation'] = model_name

        return df

    else:
        print("The mixed model parameter should be YES or NO and you have assigned: ", model_mixed)
        return None


In [32]:
def get_explanation_bedrock(df: pd.DataFrame, model_name: str, model_mixed: str):
    model_mixed = model_mixed.upper()



    if model_mixed == "YES" or model_mixed == "NO":
        start_time = time.time()
        encoding = tiktoken.get_encoding("cl100k_base")

        # Create a Bedrock runtime client
        bedrock = session.client(service_name='bedrock-runtime')

        # System prompt
        system_prompt = "You are an expert in citation analysis. Your task is to judge whether provenances contribute full, partial, or no support for their associated **generated sentence**."
        system_messages = [{"text": system_prompt}]

        
        for i in range(len(df)):
            row_df = df.iloc[[i]]
            if row_df["Has_Provenance"].values[0] == "No":
                df.at[i,'Response_Citation']="N/A"
                df.at[i,'Explanation_Citation']="N/A"
                df.at[i, 'Citation_Precision'] = "N/A"
                #df.at[i,'Clauses']="N/A"
                df.at[i,"Thinking_Citation"] = "N/A"
                df.at[i,"Is_Mixed_Model_Citation"] = "N/A"
                df.at[i,"Context_Window_Citation"] = 0
                df.at[i,"Time_Citation"] = 0
                df.at[i,"Model_Citation"] = model_name
            else:

                if "mistral-large" in model_name:
                    user_prompt = prompt_mistral.replace("_content_", generate_xml_sentence(row_df))
                else:
                    user_prompt = prompt.replace("_content_", generate_xml_sentence(row_df))

                num_tokens = len(encoding.encode(user_prompt))
                print(f"\n Model Name: {model_name} - Context Window: {num_tokens} tokens \n")
                print(user_prompt)

                user_messages = [{"role": "user", "content": [{"text": user_prompt}]}]

                thinking = None
                response_final = None

                try:
                    if "claude-3-7-sonnet" in model_name and model_mixed == "YES":
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 5000, # Max 131072
                                "temperature": 1,
                            },
                            additionalModelRequestFields={
                                'thinking': {
                                    "type": "enabled",
                                    "budget_tokens": 2000
                                }
                            }
                        )
                    elif "claude-3-7-sonnet" in model_name and model_mixed == "NO":
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 5000, # Max 131072
                                "temperature": 0,
                                "topP": 0
                            }
                        )
                    elif "deepseek.r1" in model_name:
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 32000, # Max 32768
                                "temperature": 0.0, # 0.7
                                "topP": 0.0 #0.7
                            },
                        )
                    elif "mistral-large" in model_name:
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 8000, # Max 8000
                                "temperature": 0,
                                "topP": 0.0
                            },
                        )
                    elif "llama3-3" in model_name:
                        response = bedrock.converse(
                            modelId=model_name,
                            system=system_messages,
                            messages=user_messages,
                            inferenceConfig={
                                "maxTokens": 128000,
                                "temperature": 0,
                                "topP": 0.0
                            },
                        )

                    # Extract content from response
                    contents = response["output"]["message"]["content"]

                    for content in contents:
                        if "reasoningContent" in content:
                            thinking = content["reasoningContent"]["reasoningText"]["text"]
                        if "text" in content:
                            response_final = content["text"]

                    end_time = time.time()
                    elapsed_time = round(end_time - start_time, 2)
                    print(f"\nTotal execution time: {elapsed_time} seconds")
                    print("Response: \n", response_final)

                    df.at[i,'Response_Citation']=response_final
                    df.at[i,'Explanation_Citation']=extract_explanation(response_final)
                    df.at[i, 'Citation_Precision'] = extract_citation_precision(response_final)
                    df.at[i,"Thinking_Citation"] = thinking
                    df.at[i,"Is_Mixed_Model_Citation"] = model_mixed
                    df.at[i,"Context_Window_Citation"] = num_tokens
                    df.at[i,"Time_Citation"] = elapsed_time
                    df.at[i,"Model_Citation"] = model_name

                except Exception as e:
                    print(f"An error occurred: {str(e)}")
                    continue

        return df

    else:
        print("The mixed model parameter should be YES or NO and you have assigned: ", model_mixed)
        return None

In [33]:
def get_explanation(df: pd.DataFrame, model_name: str, model_mixed: str):
    
    if "deepseek.r1" in model_name or "anthropic.claude-3-7-sonnet" in model_name or "mistral-large" in model_name or "llama3-3" in model_name:
        response_df=get_explanation_bedrock(df,model_name,model_mixed)  
    else:
        response_df=get_explanation_others(df,model_name,model_mixed)
    
    return response_df

In [34]:
# generated_sentence_final1=generated_sentence_final.head(5)
# generated_sentence_final1

In [35]:
# models = ["gpt-4.1-2025-04-14","us.deepseek.r1-v1:0"]
# models_mixed=["No","No"]

models = ["gpt-4.1-2025-04-14"]
# models = ["us.deepseek.r1-v1:0"]
models_mixed=["No"]

all_sentences_explanation = pd.DataFrame()

# Loop through the list and print each name
for model,model_mixed in zip(models,models_mixed):
    
    sentences_explanation= get_explanation(generated_sentence_final, model,model_mixed)
    all_sentences_explanation= pd.concat([all_sentences_explanation, sentences_explanation], ignore_index=True)




 Model Name: gpt-4.1-2025-04-14 - Context Window: 983 tokens 


<context>  
You are an expert in citation analysis. Your task is to judge whether provenances contribute full, partial, or no support for their associated generated sentence.  
</context>  

<generated_sentence>
Research on the citation of retracted publications has focused on understanding the mechanisms by which unreliable work continues to influence the scientific literature, with approaches developed for auditing literature that cites unreliable work, based on the recognition that not all citations are equally influential in citing publications [22] 
</generated_sentence>


<source_details>
   <source>
      <provenance>(339,1)</provenance>
      <text>To identify publications that propagate unreliability, researchers have developed approaches for auditing literature that cites unreliable work, relying on the fact that not all citations are equally influential in citing publications [22].</text>
   </source>
</source_

In [37]:
all_sentences_explanation 

,Title,Paragraph,Generated_Sentence,Provenance,Id_Order,Has_Provenance,Source_Details,Id_Order_Provenance,Id_Provenance,Final_Sentence,...,Time_Transformation,Model_Transformation,Response_Citation,Explanation_Citation,Citation_Precision,Thinking_Citation,Is_Mixed_Model_Citation,Context_Window_Citation,Time_Citation,Model_Citation
0,1. Citation of Retracted Publications and Epis...,1,Research on the citation of retracted publicat...,"(339,1)",1,Yes,"[{'Provenance': '(339,1)', 'Paper_Title': 'Add...",1.0,(Prov.1),Research on the citation of retracted publicat...,...,1.41,gpt-4.1-2025-04-14,<response> \n <citation_precision>Partial S...,1) Overall trustworthiness assessment: \n ...,Partial Support,NA,NO,983.0,6.44,gpt-4.1-2025-04-14
1,1. Citation of Retracted Publications and Epis...,1,"In parallel, several citation management tools...","(339,12)",2,Yes,"[{'Provenance': '(339,12)', 'Paper_Title': 'Ad...",2.0,(Prov.2),"In parallel, several citation management tools...",...,3.20,gpt-4.1-2025-04-14,<response> \n <citation_precision>Full Supp...,1) Overall trustworthiness assessment: \n ...,Full Support,NA,NO,949.0,11.87,gpt-4.1-2025-04-14
2,1. Citation of Retracted Publications and Epis...,1,Post-retraction citation was first studied in ...,NaN,3,No,NaN,NaN,NaN,Post-retraction citation was first studied in ...,...,0.00,gpt-4.1-2025-04-14,N/A,N/A,N/A,N/A,N/A,0.0,0.00,gpt-4.1-2025-04-14
3,1. Citation of Retracted Publications and Epis...,1,g,NaN,4,No,NaN,NaN,NaN,g,...,0.00,gpt-4.1-2025-04-14,N/A,N/A,N/A,N/A,N/A,0.0,0.00,gpt-4.1-2025-04-14
4,1. Citation of Retracted Publications and Epis...,1,", [212] [41] [10]); the most comprehensive ide...","(8,5)",5,Yes,"[{'Provenance': '(8,5)', 'Paper_Title': 'Conti...",3.0,(Prov.3),", [212] [41] [10]); the most comprehensive ide...",...,4.55,gpt-4.1-2025-04-14,<response> \n <citation_precision>Partial S...,1) Overall trustworthiness assessment: \n ...,Partial Support,NA,NO,1049.0,22.31,gpt-4.1-2025-04-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,3. Argumentation Theory and Curation in Scient...,2,One well-known example of provenance-based cur...,"(45,13)",62,Yes,"[{'Provenance': '(45,13)', 'Paper_Title': 'Tow...",50.0,(Prov.50),One well-known example of provenance-based cur...,...,57.75,gpt-4.1-2025-04-14,<response> \n <citation_precision>Full Supp...,1) Overall trustworthiness assessment: \n ...,Full Support,NA,NO,965.0,281.28,gpt-4.1-2025-04-14
62,3. Argumentation Theory and Curation in Scient...,2,Argument scheme-based curation approaches rely...,"(45,17) (45,18)",63,Yes,"[{'Provenance': '(45,17)', 'Paper_Title': 'Tow...",51.0,(Prov.51),Argument scheme-based curation approaches rely...,...,58.93,gpt-4.1-2025-04-14,<response> \n <citation_precision>Full Supp...,1) Overall trustworthiness assessment: \n ...,Full Support,NA,NO,1025.0,285.31,gpt-4.1-2025-04-14
63,3. Argumentation Theory and Curation in Scient...,2,Such argument schemes document the underlying ...,"(45,19)",64,Yes,"[{'Provenance': '(45,19)', 'Paper_Title': 'Tow...",52.0,(Prov.52),Such argument schemes document the underlying ...,...,59.83,gpt-4.1-2025-04-14,<response> \n <citation_precision>Full Supp...,1) Overall trustworthiness assessment: \n ...,Full Support,NA,NO,957.0,288.55,gpt-4.1-2025-04-14
64,3. Argumentation Theory and Curation in Scient...,3,Argument schemes describe stereotypical reason...,"(45,2)",65,Yes,"[{'Provenance': '(45,2)', 'Paper_Title': 'Towa...",53.0,(Prov.53),Argument schemes describe stereotypical reason...,...,61.26,gpt-4.1-2025-04-14,<response> \n <citation_precision>Full Supp...,1) Overall trustworthiness assessment: \n ...,Full Support,NA,NO,931.0,293.26,gpt-4.1-2025-04-14


In [38]:
path='/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Jodi_Draft_CitationAccuracyv01_NoSentence.xlsx'
# path='/Users/francisco/Documents/Research/LRO/LRO_System/LRO_RAG_System/data/Coreference/Jodi_Draft_CitationAccuracyv01_Sentence.xlsx'

all_sentences_explanation .to_excel(path, index=False, engine='openpyxl')